# Mini Dashboards: Understanding Output Types and Callbacks

## Overview

This notebook contains **12 mini-dashboards**, each focusing on a **single output type** and how to update it through callbacks.

### What You'll Learn:
1. All possible output components in Dash
2. How to update each output type through callbacks
3. Common use cases for each output type
4. Best practices for each output

### Output Types Covered:
1. **Graph** - Interactive Plotly charts
2. **HTML Content** - Dynamic text and HTML
3. **Markdown** - Formatted text with markdown
4. **Table** - Data tables with sorting/filtering
5. **Dropdown Options** - Dynamically update dropdown choices
6. **Multiple Children** - Update multiple elements
7. **Loading State** - Show loading indicator
8. **Alerts** - Display alert messages
9. **Cards** - Update card content
10. **Badges** - Update badge values
11. **Progress Bar** - Show progress
12. **Store Component** - Store data in browser

---

## Setup

In [ ]:
!pip install dash plotly pandas dash-bootstrap-components
# Import required libraries
from dash import Dash, html, dcc, Input, Output, State, callback
import dash_bootstrap_components as dbc
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

print("✅ All libraries imported successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.0/204.0 kB 13.4 MB/s eta 0:00:00
✅ All libraries imported successfully!


## Create Sample Dataset

In [ ]:
# Create a comprehensive dataset
np.random.seed(42)

# Generate dates
dates = pd.date_range('2023-01-01', '2024-12-31', freq='D')

# Define categories
products = ['Laptop', 'Monitor', 'Keyboard', 'Mouse', 'Headphones']
regions = ['North', 'South', 'East', 'West']
categories = ['Electronics', 'Accessories']

# Generate dataset
data = []
for _ in range(500):
    product = np.random.choice(products)
    region = np.random.choice(regions)
    category = 'Electronics' if product in ['Laptop', 'Monitor'] else 'Accessories'
    date = np.random.choice(dates)
    sales = np.random.randint(100, 5000)
    quantity = np.random.randint(1, 50)

    data.append({
        'Date': date,
        'Product': product,
        'Region': region,
        'Category': category,
        'Sales': sales,
        'Quantity': quantity,
        'Profit': sales * np.random.uniform(0.2, 0.4)
    })

df = pd.DataFrame(data)
df['Month'] = df['Date'].dt.to_period('M').astype(str)
df['Year'] = df['Date'].dt.year

print(f"Dataset created with {len(df)} records")
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
df.head()

Dataset created with 500 records
Date range: 2023-01-02 00:00:00 to 2024-12-31 00:00:00


,Date,Product,Region,Category,Sales,Quantity,Profit,Month,Year
0,2023-09-28,Mouse,North,Accessories,3872,21,895.220835,2023-09,2023
1,2023-11-27,Keyboard,East,Accessories,3544,36,810.064001,2023-11,2023
2,2023-11-05,Keyboard,South,Accessories,869,24,318.478531,2023-11,2023
3,2023-10-04,Monitor,West,Electronics,1284,12,413.872532,2023-10,2023
4,2024-07-14,Headphones,West,Accessories,574,42,120.157218,2024-07,2024


---

# Output Type 1: Graph

## Concept:
The **Graph** output displays interactive Plotly charts. This is the most common output type.

## Properties You Can Update:
- `figure` - The entire chart
- `style` - CSS styling
- `className` - CSS classes

## Use Cases:
- Display charts based on user input
- Update chart data dynamically
- Change chart type based on selection

In [ ]:
# Output Type 1: Graph
app1 = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app1.layout = dbc.Container([
    html.H1("Output Type 1: Graph", className="mb-4 mt-4"),

    dbc.Row([
        dbc.Col([
            html.Label("Select Chart Type:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='chart-type-dropdown',
                options=[
                    {'label': 'Bar Chart', 'value': 'bar'},
                    {'label': 'Line Chart', 'value': 'line'},
                    {'label': 'Pie Chart', 'value': 'pie'},
                    {'label': 'Scatter Plot', 'value': 'scatter'}
                ],
                value='bar',
                clearable=False
            )
        ], md=4)
    ], className="mb-4"),

    dbc.Row([
        dbc.Col([
            dcc.Graph(id='output-graph')
        ])
    ])
], fluid=True)

@app1.callback(
    Output('output-graph', 'figure'),
    Input('chart-type-dropdown', 'value')
)
def update_graph(chart_type):
    product_sales = df.groupby('Product')['Sales'].sum().reset_index().sort_values('Sales', ascending=False)

    if chart_type == 'bar':
        fig = px.bar(product_sales, x='Product', y='Sales', title='Sales by Product')
    elif chart_type == 'line':
        monthly = df.groupby('Month')['Sales'].sum().reset_index()
        fig = px.line(monthly, x='Month', y='Sales', title='Monthly Sales', markers=True)
    elif chart_type == 'pie':
        fig = px.pie(product_sales, names='Product', values='Sales', title='Sales Distribution')
    else:  # scatter
        fig = px.scatter(df.sample(100), x='Quantity', y='Sales', color='Product', title='Quantity vs Sales')

    fig.update_layout(height=400)
    return fig

print("✅ Output Type 1 (Graph) created!")
print("To run: app1.run(debug=True, port=8080)")

✅ Output Type 1 (Graph) created!
To run: app1.run(debug=True, port=8080)


In [ ]:
app1.run(jupyter_mode="inline", jupyter_height=900)

<IPython.core.display.Javascript object>

---

# Output Type 2: HTML Content

## Concept:
Update **HTML elements** like `html.Div`, `html.H1`, `html.P`, etc. This allows you to dynamically change text, styling, and structure.

## Properties You Can Update:
- `children` - Content inside the element
- `style` - CSS styling
- `className` - CSS classes

## Use Cases:
- Display dynamic text
- Show calculation results
- Update status messages

In [ ]:
# Output Type 2: HTML Content
app2 = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app2.layout = dbc.Container([
    html.H1("Output Type 2: HTML Content", className="mb-4 mt-4"),

    dbc.Row([
        dbc.Col([
            html.Label("Select a Product:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='product-dropdown-2',
                options=[{'label': p, 'value': p} for p in df['Product'].unique()],
                value='Laptop'
            )
        ], md=4)
    ], className="mb-4"),

    dbc.Row([
        dbc.Col([
            html.Div(id='output-html-content', style={
                'padding': '20px',
                'backgroundColor': '#f0f0f0',
                'borderRadius': '5px',
                'fontSize': '18px'
            })
        ])
    ])
], fluid=True)

@app2.callback(
    Output('output-html-content', 'children'),
    Input('product-dropdown-2', 'value')
)
def update_html(selected_product):
    filtered = df[df['Product'] == selected_product]
    total_sales = filtered['Sales'].sum()
    avg_sales = filtered['Sales'].mean()
    count = len(filtered)
    '''
    return html.Div([
        html.H3(f"📊 {selected_product} Statistics", style={'color': '#2c3e50'}),
        html.Hr(),
        html.P(f"Total Sales: ${total_sales:,.0f}", style={'fontSize': '18px', 'color': '#27ae60'}),
        html.P(f"Average Sale: ${avg_sales:,.0f}", style={'fontSize': '18px', 'color': '#3498db'}),
        html.P(f"Number of Transactions: {count}", style={'fontSize': '18px', 'color': '#e74c3c'})
    ])
    '''

    # ---- Return 4 cards in a single row ----
    return dbc.Row([
        dbc.Col(
            dbc.Card([
                dbc.CardBody([
                    html.H5("Product", className="card-title text-success"),
                    html.H3(f"{selected_product}", className="card-text")
                ])
            ], className="shadow-sm"),
            md=3
        ),
        dbc.Col(
            dbc.Card([
                dbc.CardBody([
                    html.H5("Total Sales", className="card-title text-primary"),
                    html.H3(f"${total_sales:,.0f}", className="card-text")
                ])
            ], className="shadow-sm"),
            md=3
        ),
        dbc.Col(
            dbc.Card([
                dbc.CardBody([
                    html.H5("Average Sale", className="card-title text-info"),
                    html.H3(f"${avg_sales:,.0f}", className="card-text")
                ])
            ], className="shadow-sm"),
            md=3
        ),
        dbc.Col(
            dbc.Card([
                dbc.CardBody([
                    html.H5("Transactions", className="card-title text-danger"),
                    html.H3(f"{count}", className="card-text")
                ])
            ], className="shadow-sm"),
            md=3
        )

    ], className="g-3")   # g-3 adds spacing between cols

print("✅ Output Type 2 (HTML Content) created!")
print("To run: app2.run(debug=True, port=8081)")

✅ Output Type 2 (HTML Content) created!
To run: app2.run(debug=True, port=8081)


In [ ]:
app2.run(jupyter_mode="inline", jupyter_height=900)

<IPython.core.display.Javascript object>

---

# Output Type 3: Markdown

## Concept:
The **Markdown** component allows you to display formatted text using markdown syntax.

## Properties You Can Update:
- `children` - Markdown formatted text

## Use Cases:
- Display formatted reports
- Show descriptions with formatting
- Create dynamic documentation

In [ ]:
# Output Type 3: Markdown
app3 = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app3.layout = dbc.Container([
    html.H1("Output Type 3: Markdown", className="mb-4 mt-4"),

    dbc.Row([
        dbc.Col([
            html.Label("Select a Region:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='region-dropdown-3',
                options=[{'label': r, 'value': r} for r in df['Region'].unique()],
                value='North'
            )
        ], md=4)
    ], className="mb-4"),

    dbc.Row([
        dbc.Col([
            dcc.Markdown(id='output-markdown')
        ])
    ])
], fluid=True)

@app3.callback(
    Output('output-markdown', 'children'),
    Input('region-dropdown-3', 'value')
)
def update_markdown(selected_region):
    filtered = df[df['Region'] == selected_region]
    total_sales = filtered['Sales'].sum()
    top_product = filtered.groupby('Product')['Sales'].sum().idxmax()
    top_sales = filtered.groupby('Product')['Sales'].sum().max()

    markdown_text = f"""
# 📍 {selected_region} Region Report

## Summary

- **Total Sales**: ${total_sales:,.0f}
- **Top Product**: {top_product}
- **Top Product Sales**: ${top_sales:,.0f}

## Key Metrics

| Metric | Value |
|--------|-------|
| Transactions | {len(filtered)} |
| Avg Sale | ${filtered['Sales'].mean():,.0f} |
| Max Sale | ${filtered['Sales'].max():,.0f} |
| Min Sale | ${filtered['Sales'].min():,.0f} |

---

**Generated**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
    """

    return markdown_text

print("✅ Output Type 3 (Markdown) created!")
print("To run: app3.run(debug=True, port=8082)")

✅ Output Type 3 (Markdown) created!
To run: app3.run(debug=True, port=8082)


In [ ]:
app3.run(jupyter_mode="inline", jupyter_height=900)

<IPython.core.display.Javascript object>

---

# Output Type 4: DataTable

## Concept:
The **DataTable** component displays data in a table format with built-in sorting, filtering, and pagination.

## Properties You Can Update:
- `data` - Table data (list of dicts)
- `columns` - Column definitions
- `style_data_conditional` - Conditional styling

## Use Cases:
- Display filtered data
- Show detailed records
- Create interactive data exploration

In [ ]:
# Output Type 4: DataTable
from dash import dash_table
app4 = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app4.layout = dbc.Container([
    html.H1("Output Type 4: DataTable", className="mb-4 mt-4"),

    dbc.Row([
        dbc.Col([
            html.Label("Select a Product:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='product-dropdown-4',
                options=[{'label': p, 'value': p} for p in df['Product'].unique()],
                value='Laptop'
            )
        ], md=4)
    ], className="mb-4"),

    dbc.Row([
        dbc.Col([
            html.Div(id='output-datatable')
        ])
    ])
], fluid=True)

@app4.callback(
    Output('output-datatable', 'children'),
    Input('product-dropdown-4', 'value')
)
def update_datatable(selected_product):
    filtered = df[df['Product'] == selected_product][['Date', 'Region', 'Sales', 'Quantity', 'Profit']].head(20)
    filtered['Date'] = filtered['Date'].dt.strftime('%Y-%m-%d')

    table = dbc.Table.from_dataframe(
        filtered,
        striped=True,
        bordered=True,
        hover=True,
        responsive=True
    )

    '''
    table = dash_table.DataTable(
        data=filtered.to_dict('records'),
        columns=[{'name': i, 'id': i} for i in filtered.columns],
        page_size=10,


    )
    '''

    return html.Div([
        html.H4(f"Recent Transactions - {selected_product}"),
        table
    ])

print("✅ Output Type 4 (DataTable) created!")
print("To run: app4.run(debug=True, port=8083)")

✅ Output Type 4 (DataTable) created!
To run: app4.run(debug=True, port=8083)


In [ ]:
app4.run(jupyter_mode="inline", jupyter_height=900)

<IPython.core.display.Javascript object>

---

# Output Type 5: Dropdown Options

## Concept:
Dynamically update the **options** of a dropdown based on another input. This is called **dependent dropdowns**.

## Properties You Can Update:
- `options` - List of options
- `value` - Selected value

## Use Cases:
- Filter dropdown options based on another selection
- Create cascading dropdowns
- Dynamic option loading

In [ ]:
# Output Type 5: Dropdown Options
app5 = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app5.layout = dbc.Container([
    html.H1("Output Type 5: Dropdown Options (Dependent Dropdowns)", className="mb-4 mt-4"),

    dbc.Row([
        dbc.Col([
            html.Label("Select a Region:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='region-dropdown-5',
                options=[{'label': r, 'value': r} for r in df['Region'].unique()],
                value='North'
            )
        ], md=4),

        dbc.Col([
            html.Label("Select a Product:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(id='product-dropdown-5')
        ], md=4)
    ], className="mb-4"),

    dbc.Row([
        dbc.Col([
            html.H5(id='output-info-5', className="mb-3")
        ])
    ])
], fluid=True)

# Callback 1: Update product options based on selected region
@app5.callback(
    Output('product-dropdown-5', 'options'),
    Input('region-dropdown-5', 'value')
)
def update_product_options(selected_region):
    products_in_region = df[df['Region'] == selected_region]['Product'].unique()
    return [{'label': p, 'value': p} for p in sorted(products_in_region)]

# Callback 2: Set default value for product dropdown
@app5.callback(
    Output('product-dropdown-5', 'value'),
    Input('product-dropdown-5', 'options')
)
def set_product_value(available_options):
    return available_options[0]['value'] if available_options else None

# Callback 3: Display info
@app5.callback(
    Output('output-info-5', 'children'),
    [Input('region-dropdown-5', 'value'),
     Input('product-dropdown-5', 'value')]
)
def update_info(region, product):
    if not product:
        return "Select a product"

    filtered = df[(df['Region'] == region) & (df['Product'] == product)]
    total_sales = filtered['Sales'].sum()

    return f"💰 {product} sales in {region} region: ${total_sales:,.0f}"

print("✅ Output Type 5 (Dropdown Options) created!")
print("To run: app5.run(debug=True, port=8084)")

✅ Output Type 5 (Dropdown Options) created!
To run: app5.run(debug=True, port=8084)


In [ ]:
app5.run(jupyter_mode="inline", jupyter_height=900)

<IPython.core.display.Javascript object>

---

# Output Type 6: Multiple Children

## Concept:
Update **multiple elements** with a single callback by returning a list of values.

## Properties You Can Update:
- Multiple `children` properties
- Multiple `style` properties
- Any combination of properties

## Use Cases:
- Update multiple metrics at once
- Synchronize multiple elements
- Create coordinated updates

In [ ]:
# Output Type 6: Multiple Children
app6 = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app6.layout = dbc.Container([
    html.H1("Output Type 6: Multiple Children (Multiple Outputs)", className="mb-4 mt-4"),

    dbc.Row([
        dbc.Col([
            html.Label("Select a Region:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='region-dropdown-6',
                options=[{'label': r, 'value': r} for r in df['Region'].unique()],
                value='North'
            )
        ], md=4)
    ], className="mb-4"),

    dbc.Row([
        dbc.Col([
            html.Div(id='metric-1', style={
                'padding': '20px',
                'backgroundColor': '#e3f2fd',
                'borderRadius': '5px',
                'marginBottom': '10px'
            })
        ], md=4),

        dbc.Col([
            html.Div(id='metric-2', style={
                'padding': '20px',
                'backgroundColor': '#e8f5e9',
                'borderRadius': '5px',
                'marginBottom': '10px'
            })
        ], md=4),

        dbc.Col([
            html.Div(id='metric-3', style={
                'padding': '20px',
                'backgroundColor': '#fff3e0',
                'borderRadius': '5px',
                'marginBottom': '10px'
            })
        ], md=4)
    ])
], fluid=True)

@app6.callback(
    [Output('metric-1', 'children'),
     Output('metric-2', 'children'),
     Output('metric-3', 'children')],
    Input('region-dropdown-6', 'value')
)
def update_multiple_metrics(selected_region):
    filtered = df[df['Region'] == selected_region]

    total_sales = filtered['Sales'].sum()
    avg_profit = filtered['Profit'].mean()
    total_transactions = len(filtered)

    metric1 = html.Div([
        html.H4("Total Sales", style={'color': '#1976d2'}),
        html.H2(f"${total_sales:,.0f}", style={'color': '#1976d2', 'fontWeight': 'bold'})
    ])

    metric2 = html.Div([
        html.H4("Avg Profit", style={'color': '#388e3c'}),
        html.H2(f"${avg_profit:,.0f}", style={'color': '#388e3c', 'fontWeight': 'bold'})
    ])

    metric3 = html.Div([
        html.H4("Transactions", style={'color': '#f57c00'}),
        html.H2(f"{total_transactions}", style={'color': '#f57c00', 'fontWeight': 'bold'})
    ])

    return metric1, metric2, metric3

print("✅ Output Type 6 (Multiple Children) created!")
print("To run: app6.run(debug=True, port=8085)")

✅ Output Type 6 (Multiple Children) created!
To run: app6.run(debug=True, port=8085)


In [ ]:
app6.run(jupyter_mode="inline", jupyter_height=900)

<IPython.core.display.Javascript object>

---

# Output Type 7: Loading State

## Concept:
The **Loading** component shows a loading indicator while a callback is executing.

## Properties You Can Update:
- `children` - Content to show when loaded

## Use Cases:
- Show loading indicator for slow operations
- Improve user experience
- Provide feedback during processing

In [ ]:
# Output Type 7: Loading State
app7 = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app7.layout = dbc.Container([
    html.H1("Output Type 7: Loading State", className="mb-4 mt-4"),

    dbc.Row([
        dbc.Col([
            html.Label("Select a Product:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='product-dropdown-7',
                options=[{'label': p, 'value': p} for p in df['Product'].unique()],
                value='Laptop'
            )
        ], md=4)
    ], className="mb-4"),

    dbc.Row([
        dbc.Col([
            dcc.Loading(
                id='loading-1',
                type='default',
                children=[
                    dcc.Graph(id='output-graph-7')
                ]
            )
        ])
    ])
], fluid=True)

@app7.callback(
    Output('output-graph-7', 'figure'),
    Input('product-dropdown-7', 'value')
)
def update_graph_with_loading(selected_product):
    import time
    time.sleep(2)  # Simulate slow operation

    filtered = df[df['Product'] == selected_product]
    monthly = filtered.groupby('Month')['Sales'].sum().reset_index()

    fig = px.line(
        monthly,
        x='Month',
        y='Sales',
        title=f'Monthly Sales - {selected_product}',
        markers=True
    )
    fig.update_layout(height=400)
    return fig

print("✅ Output Type 7 (Loading State) created!")
print("To run: app7.run(debug=True, port=8086)")

✅ Output Type 7 (Loading State) created!
To run: app7.run(debug=True, port=8086)


In [ ]:
app7.run(jupyter_mode="inline", jupyter_height=900)

<IPython.core.display.Javascript object>

---

# Output Type 8: Alert Messages

## Concept:
Display **alert messages** to inform users about results or errors.

## Properties You Can Update:
- `children` - Alert content
- `color` - Alert color (success, danger, warning, info)
- `is_open` - Show/hide alert

## Use Cases:
- Show success messages
- Display error messages
- Provide user feedback

In [ ]:
# Output Type 8: Alert Messages
app8 = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app8.layout = dbc.Container([
    html.H1("Output Type 8: Alert Messages", className="mb-4 mt-4"),

    dbc.Row([
        dbc.Col([
            html.Label("Select a Region:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='region-dropdown-8',
                options=[{'label': r, 'value': r} for r in df['Region'].unique()],
                value='North'
            )
        ], md=4)
    ], className="mb-4"),

    dbc.Row([
        dbc.Col([
            dbc.Alert(id='output-alert-8')
        ])
    ])
], fluid=True)

@app8.callback(
    Output('output-alert-8', 'children'),
    Input('region-dropdown-8', 'value')
)
def update_alert(selected_region):
    filtered = df[df['Region'] == selected_region]
    total_sales = filtered['Sales'].sum()

    return f"✅ Successfully loaded data for {selected_region} region. Total sales: ${total_sales:,.0f}"

print("✅ Output Type 8 (Alert Messages) created!")
print("To run: app8.run(debug=True, port=8087)")

✅ Output Type 8 (Alert Messages) created!
To run: app8.run(debug=True, port=8087)


In [ ]:
app8.run(jupyter_mode="inline", jupyter_height=900)

<IPython.core.display.Javascript object>

---

# Output Type 9: Card Content

## Concept:
Update the **content of cards** to display dynamic information.

## Properties You Can Update:
- `children` - Card content
- `style` - Card styling

## Use Cases:
- Display KPI cards
- Show metric summaries
- Create dynamic dashboards

In [ ]:
# Output Type 9: Card Content
app9 = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app9.layout = dbc.Container([
    html.H1("Output Type 9: Card Content", className="mb-4 mt-4"),

    dbc.Row([
        dbc.Col([
            html.Label("Select a Product:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='product-dropdown-9',
                options=[{'label': p, 'value': p} for p in df['Product'].unique()],
                value='Laptop'
            )
        ], md=4)
    ], className="mb-4"),

    dbc.Row([
        dbc.Col([
            dbc.Card(id='output-card-9')
        ], md=6)
    ])
], fluid=True)

@app9.callback(
    Output('output-card-9', 'children'),
    Input('product-dropdown-9', 'value')
)
def update_card(selected_product):
    filtered = df[df['Product'] == selected_product]
    total_sales = filtered['Sales'].sum()
    avg_sale = filtered['Sales'].mean()
    max_sale = filtered['Sales'].max()

    return dbc.CardBody([
        html.H4(f"📊 {selected_product} Summary", className="card-title"),
        html.Hr(),
        html.P([html.Strong("Total Sales: "), f"${total_sales:,.0f}"], className="card-text"),
        html.P([html.Strong("Average Sale: "), f"${avg_sale:,.0f}"], className="card-text"),
        html.P([html.Strong("Maximum Sale: "), f"${max_sale:,.0f}"], className="card-text"),
        html.P([html.Strong("Transactions: "), f"{len(filtered)}"], className="card-text")
    ])

print("✅ Output Type 9 (Card Content) created!")
print("To run: app9.run(debug=True, port=8088)")

✅ Output Type 9 (Card Content) created!
To run: app9.run(debug=True, port=8088)


In [ ]:
app9.run(jupyter_mode="inline", jupyter_height=900)

<IPython.core.display.Javascript object>

---

# Output Type 10: Badge Content

## Concept:
Update **badge** content to display counts or status indicators.

## Properties You Can Update:
- `children` - Badge text
- `color` - Badge color

## Use Cases:
- Display counts
- Show status indicators
- Create notification badges

In [ ]:
# Output Type 10: Badge Content
app10 = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app10.layout = dbc.Container([
    html.H1("Output Type 10: Badge Content", className="mb-4 mt-4"),

    dbc.Row([
        dbc.Col([
            html.Label("Select a Region:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='region-dropdown-10',
                options=[{'label': r, 'value': r} for r in df['Region'].unique()],
                value='North'
            )
        ], md=4)
    ], className="mb-4"),

    dbc.Row([
        dbc.Col([
            html.H5([
                "Total Transactions: ",
                dbc.Badge(id='output-badge-10', color="primary")
            ])
        ], md=4),

        dbc.Col([
            html.H5([
                "Top Product: ",
                dbc.Badge(id='output-badge-product', color="success")
            ])
        ], md=4)
    ])
], fluid=True)

@app10.callback(
    [Output('output-badge-10', 'children'),
     Output('output-badge-product', 'children')],
    Input('region-dropdown-10', 'value')
)
def update_badges(selected_region):
    filtered = df[df['Region'] == selected_region]
    transaction_count = len(filtered)
    top_product = filtered.groupby('Product')['Sales'].sum().idxmax()

    return transaction_count, top_product

print("✅ Output Type 10 (Badge Content) created!")
print("To run: app10.run(debug=True, port=8089)")

✅ Output Type 10 (Badge Content) created!
To run: app10.run(debug=True, port=8089)


In [ ]:
app10.run(jupyter_mode="inline", jupyter_height=900)

<IPython.core.display.Javascript object>

---

# Output Type 11: Progress Bar

## Concept:
Update **progress bars** to show completion status or metrics.

## Properties You Can Update:
- `value` - Progress percentage
- `children` - Progress label

## Use Cases:
- Show progress towards goals
- Display completion percentage
- Visualize metrics

In [ ]:
# Output Type 11: Progress Bar
app11 = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app11.layout = dbc.Container([
    html.H1("Output Type 11: Progress Bar", className="mb-4 mt-4"),

    dbc.Row([
        dbc.Col([
            html.Label("Select a Product:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='product-dropdown-11',
                options=[{'label': p, 'value': p} for p in df['Product'].unique()],
                value='Laptop'
            )
        ], md=4)
    ], className="mb-4"),

    dbc.Row([
        dbc.Col([
            html.H5("Sales Progress (% of Max)"),
            dbc.Progress(id='output-progress-11', style={'height': '30px'})
        ])
    ])
], fluid=True)

@app11.callback(
    Output('output-progress-11', 'value'),
    Input('product-dropdown-11', 'value')
)
def update_progress(selected_product):
    filtered = df[df['Product'] == selected_product]
    total_sales = filtered['Sales'].sum()
    max_sales = df.groupby('Product')['Sales'].sum().max()

    progress_value = int((total_sales / max_sales) * 100)
    return progress_value

print("✅ Output Type 11 (Progress Bar) created!")
print("To run: app11.run(debug=True, port=8090)")

✅ Output Type 11 (Progress Bar) created!
To run: app11.run(debug=True, port=8090)


In [ ]:
app11.run(jupyter_mode="inline", jupyter_height=900)

<IPython.core.display.Javascript object>

---

# Output Type 12: Store Component

## Concept:
The **Store** component stores data in the browser (client-side). Useful for caching data and passing between callbacks.

## Properties You Can Update:
- `data` - Stored data (JSON serializable)

## Use Cases:
- Cache filtered data
- Pass data between callbacks
- Reduce server load
- Store user selections

In [ ]:
# Output Type 12: Store Component
app12 = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app12.layout = dbc.Container([
    html.H1("Output Type 12: Store Component", className="mb-4 mt-4"),

    # Store component (invisible)
    dcc.Store(id='stored-data'),

    dbc.Row([
        dbc.Col([
            html.Label("Select a Region:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='region-dropdown-12',
                options=[{'label': r, 'value': r} for r in df['Region'].unique()],
                value='North'
            )
        ], md=4)
    ], className="mb-4"),

    dbc.Row([
        dbc.Col([
            html.H5("Data stored in browser:"),
            html.Pre(id='output-stored-data', style={
                'backgroundColor': '#f5f5f5',
                'padding': '15px',
                'borderRadius': '5px',
                'maxHeight': '300px',
                'overflow': 'auto'
            })
        ])
    ])
], fluid=True)

# Callback 1: Store data
@app12.callback(
    Output('stored-data', 'data'),
    Input('region-dropdown-12', 'value')
)
def store_data(selected_region):
    filtered = df[df['Region'] == selected_region]

    # Convert to JSON-serializable format
    data_to_store = {
        'region': selected_region,
        'total_sales': float(filtered['Sales'].sum()),
        'transaction_count': len(filtered),
        'avg_sale': float(filtered['Sales'].mean()),
        'timestamp': datetime.now().isoformat()
    }

    return data_to_store

# Callback 2: Display stored data
@app12.callback(
    Output('output-stored-data', 'children'),
    Input('stored-data', 'data')
)
def display_stored_data(stored_data):
    if stored_data:
        import json
        return json.dumps(stored_data, indent=2)
    return "No data stored yet"

print("✅ Output Type 12 (Store Component) created!")
print("To run: app12.run(debug=True, port=8091)")

✅ Output Type 12 (Store Component) created!
To run: app12.run(debug=True, port=8091)


In [ ]:
app12.run(jupyter_mode="inline", jupyter_height=900)

<IPython.core.display.Javascript object>

---

# Summary: Output Types and Callbacks

## Quick Reference Table

| Output Type | Component | Property | Use Case |
|-------------|-----------|----------|----------|
| **Graph** | `dcc.Graph` | `figure` | Display interactive charts |
| **HTML Content** | `html.Div`, `html.H1`, etc. | `children` | Dynamic text and HTML |
| **Markdown** | `dcc.Markdown` | `children` | Formatted text |
| **DataTable** | `dash.dash_table.DataTable` | `data`, `columns` | Interactive tables |
| **Dropdown Options** | `dcc.Dropdown` | `options` | Dependent dropdowns |
| **Multiple Children** | Any component | Multiple properties | Update multiple elements |
| **Loading State** | `dcc.Loading` | `children` | Show loading indicator |
| **Alert** | `dbc.Alert` | `children`, `color` | Display messages |
| **Card** | `dbc.Card` | `children` | Dynamic card content |
| **Badge** | `dbc.Badge` | `children`, `color` | Display counts/status |
| **Progress Bar** | `dbc.Progress` | `value` | Show progress |
| **Store** | `dcc.Store` | `data` | Cache data in browser |

---

## Callback Patterns for Outputs

### Pattern 1: Update Single Property
```python
@app.callback(
    Output('graph', 'figure'),
    Input('dropdown', 'value')
)
def update(value):
    return figure
```

### Pattern 2: Update Multiple Properties of Same Component
```python
@app.callback(
    [Output('alert', 'children'),
     Output('alert', 'color')],
    Input('button', 'n_clicks')
)
def update(n_clicks):
    return message, color
```

### Pattern 3: Update Multiple Components
```python
@app.callback(
    [Output('graph', 'figure'),
     Output('info', 'children'),
     Output('badge', 'children')],
    Input('dropdown', 'value')
)
def update(value):
    return figure, info_text, count
```

### Pattern 4: Dependent Outputs
```python
@app.callback(
    Output('dropdown2', 'options'),
    Input('dropdown1', 'value')
)
def update_options(value1):
    return options_list
```

---

## Key Concepts

### Output Properties
- **`children`** - Content inside component (most common)
- **`figure`** - Plotly figure (for Graph)
- **`value`** - Selected value (for inputs)
- **`options`** - Available options (for dropdowns)
- **`data`** - Stored data (for Store)
- **`style`** - CSS styling
- **`className`** - CSS classes

### Best Practices
1. **Keep callbacks focused** - One logical operation per callback
2. **Use appropriate output types** - Choose the right component for the job
3. **Provide feedback** - Show loading states and messages
4. **Cache data** - Use Store for expensive computations
5. **Handle edge cases** - Empty selections, missing data, etc.
6. **Optimize performance** - Avoid unnecessary updates
7. **Test thoroughly** - Verify all output types work correctly


---


---

## Resources

- [Dash Callback Documentation](https://dash.plotly.com/basic-callbacks)
- [Dash Core Components](https://dash.plotly.com/dash-core-components)
- [Dash Bootstrap Components](https://dash-bootstrap-components.opensource.faculty.ai/)
- [Plotly Chart Types](https://plotly.com/python/)